<a href="https://colab.research.google.com/github/m-usmanmalik/Artificial-Intelligence/blob/main/AILab4/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
import networkx as nx
import matplotlib.pyplot as plt

# ---------- Same tree as before ----------
tree = {
    'A': ['B', 'C', 'D'],
    'B': ['E', 'F'],
    'C': ['G'],
    'D': ['H', 'I'],
    'E': ['J', 'K'],
    'F': ['L'],
    'G': ['M'],
    'H': ['N'],
    'I': ['O'],
    'J': [], 'K': [], 'L': [], 'M': [], 'N': [], 'O': []
}

# ---------- Build the networkx graph ----------
G = nx.DiGraph()
for parent, children in tree.items():
    G.add_node(parent)
    for child in children:
        G.add_edge(parent, child)


# ---------- Tree layout (root on top, no Graphviz needed) ----------
def tree_layout(tree, root):
    pos, next_x = {}, [0]

    def place(node, depth):
        children = tree[node]
        if not children:                      # leaf: take next free x slot
            x = next_x[0]
            next_x[0] += 1
        else:                                 # parent: centre above children
            xs = [place(c, depth + 1) for c in children]
            x = sum(xs) / len(xs)
        pos[node] = (x, -depth)
        return x

    place(root, 0)
    return pos


# ---------- Depth-limited DFS that records visited nodes ----------
def dls(node, goal, limit, visited):
    visited.add(node)
    if node == goal:
        return True
    if limit == 0:
        return False
    for child in tree[node]:
        if dls(child, goal, limit - 1, visited):
            return True
    return False


# ---------- Save one image for a given depth limit ----------
def save_image(visited, depth, pos, folder):
    # colour list in the same order as G.nodes()
    colors = ['green' if node in visited else 'grey' for node in G.nodes()]

    plt.figure(figsize=(10, 6))
    nx.draw(G, pos,
            with_labels=True,
            node_color=colors,
            node_size=900,
            font_color='white',
            font_weight='bold',
            arrows=False)
    plt.title(f"IDDFS - depth limit {depth}  "
              f"({len(visited)} nodes visited)")
    filename = os.path.join(folder, f"iddfs_depth_{depth}.png")
    plt.savefig(filename, dpi=120, bbox_inches='tight')
    plt.close()
    return filename


# ---------- IDDFS with visualisation ----------
def iddfs_visualise(start, goal, max_depth=10, folder="iddfs_images"):
    os.makedirs(folder, exist_ok=True)
    pos = tree_layout(tree, start)

    for depth in range(max_depth + 1):
        visited = set()
        found = dls(start, goal, depth, visited)
        file = save_image(visited, depth, pos, folder)
        print(f"Depth {depth}: visited {len(visited)} nodes -> saved {file}")
        if found:
            print(f"Goal '{goal}' found at depth limit {depth}.")
            return
    print("Goal not found.")


if __name__ == "__main__":
    iddfs_visualise('A', 'M')


Depth 0: visited 1 nodes -> saved iddfs_images/iddfs_depth_0.png
Depth 1: visited 4 nodes -> saved iddfs_images/iddfs_depth_1.png
Depth 2: visited 9 nodes -> saved iddfs_images/iddfs_depth_2.png
Depth 3: visited 10 nodes -> saved iddfs_images/iddfs_depth_3.png
Goal 'M' found at depth limit 3.
